# Ethiopia Climate EDA

## Business Objective
EthioClimate Analytics is supporting Ethiopia's COP32 preparations by translating NASA POWER climate observations into evidence-backed insights for policy and negotiation. This notebook focuses on Ethiopia's historical climate patterns from 2015 to March 2026 and is structured to answer three layers of analysis: what is changing, what that may mean in practice, and what it suggests for adaptation and climate finance priorities.

## Evidence-Backed Insight Ladder
- **EDA**: identify trends, seasonal cycles, and anomalies.
- **Report**: connect trends to likely impacts using a secondary source if needed.
- **Position paper**: translate findings into a policy or finance ask.

## References to Self-Learn
- NASA POWER documentation for dataset structure and sentinel values.
- pandas documentation for `read_csv`, date parsing, grouping, and missing-data handling.
- SciPy documentation for `scipy.stats.zscore`.
- seaborn and matplotlib documentation for statistical graphics.

## 1. Data Loading and Date Parsing
Load `ethiopia.csv`, add the `Country` column, parse `YEAR` and `DOY` into a proper date, and extract a `Month` column for seasonal analysis.

The data file is expected to sit at the workspace root. The cleaned export will be written to `data/ethiopia_clean.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import zscore

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

In [ ]:
country = "ethiopia"
raw_path = Path(f"{country}.csv")
clean_path = Path("data") / f"{country}_clean.csv"

df = pd.read_csv(raw_path)
df["Country"] = country.title()
df["DATE"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["Month"] = df["DATE"].dt.month
df["Month_Name"] = df["DATE"].dt.month_name()

weather_columns = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "T2M_RANGE",
    "PRECTOTCORR",
    "RH2M",
    "WS2M",
    "WS2M_MAX",
    "PS",
    "QV2M",
]

df = df.replace(-999, np.nan)

print(f"Loaded rows: {len(df):,}")
df.head()

## 2. Summary Statistics and Missing-Value Report
Replace all `-999` sentinel values with `np.nan` before any statistics so the summary reflects true missingness. Then check for duplicate rows, describe the numeric columns, and compute missing-value percentages by column.

Write your interpretation directly under each output:
- State how many duplicate rows were found and confirm they are exact row-level duplicates across the full record.
- Briefly describe the central tendency, spread, and any unusual min/max values in `df.describe()`.
- List every column with more than 5% missing values and explain how that could weaken trend, correlation, or seasonal analysis.
- If missingness is concentrated in a single variable family, note whether that suggests a sensor/data-collection issue or a real seasonal gap.

In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_columns = list(df.columns)

df = df.drop_duplicates().copy()

numeric_summary = df.select_dtypes(include=["number"]).describe().T
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct,
})
high_missing = missing_report[missing_report["missing_pct"] > 5]

print(f"Duplicate rows found: {duplicate_count}")
print("Duplicate rows are exact row-level matches across these columns:")
print(duplicate_columns)
print("Numeric summary:")
display(numeric_summary)
print("Missing-value report:")
display(missing_report)
print("Columns above 5% missing:")
display(high_missing)

## 3. Outlier Detection and Cleaning Decision
Use z-scores on the main weather variables to flag extreme daily observations. In this notebook, we flag rows where any selected variable has |Z| > 3, but we retain those rows unless later inspection shows an impossible value or a clear data-entry error. That choice preserves genuine climate extremes, which are often the most policy-relevant observations.

This matters because climate analysis should preserve genuine extremes unless there is evidence they are impossible values or measurement glitches. If a later review shows a suspicious spike, capping or dropping should only be done with a documented reason.

In [ ]:
zscore_columns = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]
zscore_frame = df[zscore_columns].apply(lambda series: zscore(series, nan_policy="omit"))
outlier_mask = zscore_frame.abs().gt(3).any(axis=1)
outlier_count = int(outlier_mask.sum())
outlier_by_column = zscore_frame.abs().gt(3).sum().sort_values(ascending=False)
df["Outlier_Flag"] = outlier_mask

print(f"Rows flagged as outliers: {outlier_count}")
display(outlier_by_column.to_frame(name="flagged_rows"))
df[outlier_mask].head()

## 4. Handling Missing Values
Drop rows with more than 30% missing values, then forward-fill weather variables after sorting by date. This keeps the daily time series usable while avoiding over-imputation in rows that are too sparse to trust. After cleaning, export the result to `data/ethiopia_clean.csv`.

Document any variables that still remain missing after this step and explain whether they are safe to keep for the chosen charts. The `data/` folder stays ignored by Git, so the cleaned CSV will not be committed.

In [ ]:
row_missing_share = df.isna().mean(axis=1)
df_clean = df.loc[row_missing_share <= 0.30].copy()
df_clean = df_clean.sort_values("DATE")
df_clean[weather_columns] = df_clean[weather_columns].ffill()

Path("data").mkdir(exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print(f"Cleaned rows saved to: {clean_path}")
print(f"Rows removed for excessive missingness: {int((row_missing_share > 0.30).sum())}")
print(f"Remaining missing values: {int(df_clean.isna().sum().sum())}")
df_clean.head()

## 5. Time Series Analysis
Plot monthly average temperature and monthly total precipitation across the full period. Annotate the warmest and coolest months, plus the peak rainy season months. Then add a short markdown note below the charts describing any visible seasonality, spikes, or unusual breaks in the pattern.

In [ ]:
monthly = df_clean.resample("M", on="DATE").agg({
    "T2M": "mean",
    "PRECTOTCORR": "sum",
}).reset_index()

warmest_row = monthly.loc[monthly["T2M"].idxmax()]
coolest_row = monthly.loc[monthly["T2M"].idxmin()]
rainiest_rows = monthly.nlargest(3, "PRECTOTCORR")

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
axes[0].plot(monthly["DATE"], monthly["T2M"], color="#c44e52", linewidth=2)
axes[0].scatter([warmest_row["DATE"], coolest_row["DATE"]], [warmest_row["T2M"], coolest_row["T2M"]], color="black", zorder=5)
axes[0].annotate(f"Warmest: {warmest_row['DATE'].date()}", xy=(warmest_row["DATE"], warmest_row["T2M"]), xytext=(10, 15), textcoords="offset points")
axes[0].annotate(f"Coolest: {coolest_row['DATE'].date()}", xy=(coolest_row["DATE"], coolest_row["T2M"]), xytext=(10, -20), textcoords="offset points")
axes[0].set_title("Ethiopia Monthly Average T2M")
axes[0].set_ylabel("Temperature (°C)")

axes[1].bar(monthly["DATE"], monthly["PRECTOTCORR"], color="#4c72b0", width=20)
for _, row in rainiest_rows.iterrows():
    axes[1].annotate(f"Peak rain: {row['DATE'].date()}", xy=(row["DATE"], row["PRECTOTCORR"]), xytext=(10, 15), textcoords="offset points")
axes[1].set_title("Ethiopia Monthly Total PRECTOTCORR")
axes[1].set_ylabel("Precipitation (mm/month)")
axes[1].set_xlabel("Month")

plt.tight_layout()
plt.show()

### Time-Series Interpretation
Write 3 to 5 sentences here after running the chart. Mention whether temperature follows a stable seasonal cycle, whether rainfall is clustered into a few wetter months, and whether any unusually high or low months look like anomalies worth investigating further.

## 6. Correlation and Relationship Analysis
Inspect the correlation matrix, then use scatter plots to test the temperature-humidity and temperature-wind relationships. The three strongest correlations should be interpreted in plain language, with attention to whether they are physically intuitive or might reflect measurement linkage.

In [ ]:
numeric_df = df_clean.select_dtypes(include=["number"])
corr = numeric_df.corr()

upper_triangle = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strongest_pairs = (
    upper_triangle.stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "var_1", "level_1": "var_2"})
)
strongest_pairs["abs_correlation"] = strongest_pairs["correlation"].abs()
strongest_pairs = strongest_pairs.sort_values("abs_correlation", ascending=False).head(3)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.heatmap(corr, ax=axes[0], cmap="coolwarm", center=0, square=True)
axes[0].set_title("Correlation Heatmap")

sns.scatterplot(data=df_clean, x="T2M", y="RH2M", ax=axes[1], alpha=0.5)
axes[1].set_title("T2M vs RH2M")

sns.scatterplot(data=df_clean, x="T2M_RANGE", y="WS2M", ax=axes[2], alpha=0.5)
axes[2].set_title("T2M_RANGE vs WS2M")

plt.tight_layout()
plt.show()

print("Three strongest correlations:")
display(strongest_pairs)

### Correlation Interpretation
After running the cell above, write 3 to 5 sentences that interpret the three strongest correlations:
- Name each variable pair and state whether the relationship is positive or negative.
- Explain whether the relationship is expected from climate physics or could be influenced by data quality/seasonality.
- Note one implication for climate-risk interpretation in Ethiopia (for example, heat stress, moisture dynamics, or wind-related exposure).

## 7. Distribution Analysis
Check whether precipitation is skewed and whether a log scale is needed for a meaningful view. Then use a bubble chart to combine temperature, humidity, and precipitation into a single relationship view.

When writing up the findings, distinguish between shape, magnitude, and outlier behavior rather than treating the histogram as a standalone conclusion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
prectot = df_clean["PRECTOTCORR"].dropna()
use_log = prectot.skew() > 1
hist_data = np.log1p(prectot) if use_log else prectot
hist_label = "log1p(PRECTOTCORR)" if use_log else "PRECTOTCORR"

axes[0].hist(hist_data, bins=40, color="#55a868", edgecolor="white")
axes[0].set_title(f"Distribution of {hist_label}")
axes[0].set_xlabel(hist_label)
axes[0].set_ylabel("Frequency")

bubble = axes[1].scatter(
    df_clean["T2M"],
    df_clean["RH2M"],
    s=np.clip(df_clean["PRECTOTCORR"].fillna(0), 1, None) * 6,
    c=df_clean["PRECTOTCORR"],
    cmap="Blues",
    alpha=0.5,
)
axes[1].set_title("Bubble Chart: T2M vs RH2M")
axes[1].set_xlabel("T2M (°C)")
axes[1].set_ylabel("RH2M (%)")
fig.colorbar(bubble, ax=axes[1], label="PRECTOTCORR (mm/day)")

plt.tight_layout()
plt.show()

### Distribution Interpretation
After running the distribution cell, write 3 to 5 sentences:
- State whether PRECTOTCORR is strongly right-skewed and whether the log transform improved readability.
- Describe where most precipitation observations concentrate (low/moderate/high rainfall).
- Use the bubble chart to comment on how precipitation intensity varies across the T2M-RH2M space.

## 8. Interpretation Notes
After running the notebook, summarize the findings in three layers:
1. What changed in Ethiopia's climate patterns over time?
2. What practical climate risk or livelihood consequence might those patterns imply?
3. What policy or finance ask follows from the evidence?

For the COP32 framing, keep the final write-up focused on adaptation finance, early warning systems, drought/flood resilience, and loss-and-damage relevance where the evidence supports it.